# RecaLLM In-Domain Evaluation Analysis

Loads results from `evaluate_vllm.py` and computes summary tables and plots.

**Setup:** Run evaluations first via `submit_all.sh`, then point `RESULTS_DIR` and `models` at your output.

In [ ]:
import json
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, HTML

# ============================================================================
# Configuration — edit these for your setup
# ============================================================================

# Directory containing evaluation results.
# Expected layout: RESULTS_DIR/<model>/<dataset>/<context_length>/results.jsonl
RESULTS_DIR = "./results"

# Models to compare — directory names inside RESULTS_DIR
models = [
    # "recallm_qwen2",
    # "qwen2_baseline",
]

# Datasets grouped by category
DATASETS_BY_CATEGORY = {
    "multi_hop_qa": ["hotpotqa", "musique", "2wikimultihopqa"],
    "single_hop_qa": ["nq", "triviaqa"],
    "recall": ["retrieval", "multi_niah"],
    "reasoning_retrieval": ["math_retrieval"],
    "short_context_math": ["dapo_math", "mcqa_math"],
    "icl": ["banking77", "massive"],
    "long_doc_qa": ["quality"],
    "aggregation": ["majority_vote", "top_n_vote"],
    "reranking": ["msmarco_v2"],
    "citation_qa": ["qampari"],
}

ALL_DATASETS = [ds for cat_datasets in DATASETS_BY_CATEGORY.values() for ds in cat_datasets]

DATASET_TO_CATEGORY = {}
for cat, ds_list in DATASETS_BY_CATEGORY.items():
    for ds in ds_list:
        DATASET_TO_CATEGORY[ds] = cat

# Answer metric display names for graph titles
ANSWER_METRIC_NAMES = {
    "hotpotqa": "2-way SubEM", "musique": "2-way SubEM", "2wikimultihopqa": "2-way SubEM",
    "nq": "2-way SubEM", "triviaqa": "2-way SubEM",
    "retrieval": "SubEM", "math_retrieval": "SubEM",
    "multi_niah": "Recall",
    "dapo_math": "EM", "mcqa_math": "EM",
    "banking77": "EM", "massive": "EM",
    "quality": "Accuracy",
    "majority_vote": "EM", "top_n_vote": "Recall",
    "msmarco_v2": "NDCG@10",
    "qampari": "Citation F1 (Rec@5)",
}

# Context lengths: (display_name, token_count)
CONTEXT_LENGTHS = [
    ("4k", 4000),
    ("8k", 8000),
    ("16k", 16000),
    ("32k", 32000),
    ("64k", 64000),
    ("96k", 96000),
    ("128k", 128000),
]

# Core metrics to aggregate (must exist in results.jsonl from evaluate_vllm.py)
METRICS = [
    "score",
    "answer_score",
    "has_answer",
    "truncated",
    "gold_doc_overlap",
    "gold_doc_score",
    "used_recall",
    "n_recall_spans",
    "correct_format",
    "correct_recall_usage",
    "recall_density_multiplier",
    "n_output_ids",
    "citation_f1_plain",
    "citation_recall_plain",
    "citation_precision_plain",
]

TRUNCATE_TO = 200  # Max examples per (model, dataset, context_length)

In [ ]:
# ============================================================================
# Data Loading
# ============================================================================
# Loads results.jsonl from: RESULTS_DIR/<model>/<dataset>/<context_length>/results.jsonl
# Each line is a JSON object from evaluate_vllm.py (final_reward scoring).
#
# The loader tries multiple directory name variants for backward compatibility:
#   - 128k: also tries 120704 and 120k

# Directory name fallbacks for backward compatibility
DATASET_DIR_ALIASES = {
}
CTX_DIR_ALIASES = {
    "128k": ["128k", "120k", "120704"],
}

def find_results_path(base, model, dataset, ctx_name, ctx_val):
    """Try multiple directory name variants to find results.jsonl."""
    ds_variants = DATASET_DIR_ALIASES.get(dataset, [dataset])
    ctx_variants = CTX_DIR_ALIASES.get(ctx_name, [ctx_name, str(ctx_val)])
    for ds_dir in ds_variants:
        for ctx_dir in ctx_variants:
            path = os.path.join(base, model, ds_dir, ctx_dir, "results.jsonl")
            if os.path.exists(path):
                return path
    return None

data = []
for model in models:
    model_dir = os.path.join(RESULTS_DIR, model)
    if not os.path.isdir(model_dir):
        print(f"SKIP model dir not found: {model_dir}")
        continue
    for dataset in ALL_DATASETS:
        for ctx_name, ctx_val in CONTEXT_LENGTHS:
            result_path = find_results_path(RESULTS_DIR, model, dataset, ctx_name, ctx_val)
            if result_path is None:
                continue
            # Load args for max_new_tokens
            result_dir = os.path.dirname(result_path)
            args_path = os.path.join(result_dir, "args.json")
            max_new_tokens = None
            if os.path.exists(args_path):
                with open(args_path, "r") as f:
                    max_new_tokens = json.load(f).get("max_new_tokens")
            # Check for aborted runs
            aborted_path = os.path.join(result_dir, "aborted.txt")
            was_aborted = os.path.exists(aborted_path)
            count = 0
            with open(result_path, "r") as f:
                for line in f:
                    if TRUNCATE_TO is not None and count >= TRUNCATE_TO:
                        break
                    try:
                        rec = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    rec["model"] = model
                    rec["dataset"] = dataset
                    rec["context_length"] = ctx_val
                    rec["context_length_name"] = ctx_name
                    rec["category"] = DATASET_TO_CATEGORY.get(dataset, "unknown")
                    rec["aborted"] = was_aborted
                    if max_new_tokens is not None:
                        rec["truncated"] = rec.get("n_output_ids", 0) >= max_new_tokens
                    data.append(rec)
                    count += 1
            status = f" [ABORTED]" if was_aborted else ""
            print(f"Loaded {count:>4d}  {model}/{dataset}/{ctx_name}{status}")

if not data:
    print("\nNo data loaded. Make sure `models` list is set and result directories exist.")
else:
    df = pd.DataFrame(data)
    # Ensure metric columns exist (fill missing with NaN)
    for m in METRICS:
        if m not in df.columns:
            df[m] = np.nan

    # QAMPARI: override answer_score with plain citation F1 for fair comparison
    # (training reward blends recall-backed citation F1 + answer coverage)
    qampari_mask = df["dataset"] == "qampari"
    df.loc[qampari_mask, "answer_score"] = df.loc[qampari_mask, "citation_f1_plain"]

    pd.set_option("display.max_columns", None)
    print(f"\nLoaded {len(df):,} total examples across {df['model'].nunique()} models, "
          f"{df['dataset'].nunique()} datasets, {df['context_length'].nunique()} context lengths")
    n_aborted = df[df["aborted"]].groupby(["model", "dataset", "context_length_name"]).ngroups if "aborted" in df.columns else 0
    if n_aborted:
        print(f"  {n_aborted} run(s) were aborted early (partial results included)")
    display(df.groupby(["model", "dataset"]).size().unstack(fill_value=0))

## Grand Average Tables

Four views:
1. **Per-model grand average** -- single score per model (average across all datasets x context lengths)
2. **Per-category** -- average per model x category
3. **Per-category (score)** -- composite reward per model x category
4. **Per-dataset** -- average per model x dataset

In [ ]:
# ============================================================================
# Grand Average Tables
# ============================================================================

key_metrics = ["score", "answer_score", "gold_doc_overlap", "gold_doc_score",
               "has_answer", "truncated", "used_recall", "n_recall_spans", "n_output_ids"]

# 1. Per-model grand average
print("=" * 80)
print("PER-MODEL GRAND AVERAGE (across all datasets x context lengths)")
print("=" * 80)
grand_avg = df.groupby("model")[key_metrics].mean()
display(grand_avg.round(4))

# 2. Per-category average
print("\n" + "=" * 80)
print("PER-CATEGORY: answer_score")
print("=" * 80)
cat_pivot = df.groupby(["model", "category"])["answer_score"].mean().unstack(fill_value=np.nan)
cat_order = [c for c in DATASETS_BY_CATEGORY.keys() if c in cat_pivot.columns]
display(cat_pivot[cat_order].round(4))

print("\n" + "=" * 80)
print("PER-CATEGORY: score (composite reward)")
print("=" * 80)
cat_score_pivot = df.groupby(["model", "category"])["score"].mean().unstack(fill_value=np.nan)
cat_order_score = [c for c in DATASETS_BY_CATEGORY.keys() if c in cat_score_pivot.columns]
display(cat_score_pivot[cat_order_score].round(4))

# 3. Per-dataset average
print("\n" + "=" * 80)
print("PER-DATASET: answer_score (averaged over context lengths)")
print("=" * 80)
ds_pivot = df.groupby(["model", "dataset"])["answer_score"].mean().unstack(fill_value=np.nan)
ds_order = [d for d in ALL_DATASETS if d in ds_pivot.columns]
display(ds_pivot[ds_order].round(4))

In [ ]:
# ============================================================================
# Per-Dataset: Short-Context vs Long-Context Average (answer_score)
# ============================================================================
# Short = 4k, 8k, 16k, 32k   |   Long = 64k, 96k, 128k

SHORT_CTX = {4000, 8000, 16000, 32000}
LONG_CTX = {64000, 96000, 128000}

df_short = df[df["context_length"].isin(SHORT_CTX)]
df_long = df[df["context_length"].isin(LONG_CTX)]

short_avg = df_short.groupby(["model", "dataset"])["answer_score"].mean()
long_avg = df_long.groupby(["model", "dataset"])["answer_score"].mean()

combined = pd.DataFrame({"short": short_avg, "long": long_avg})
combined = combined.reset_index()

# Build a wide table: rows = model, columns = dataset_short, dataset_long interleaved
rows = []
for model in models:
    row = {"model": model}
    for ds in ALL_DATASETS:
        s = combined.loc[(combined["model"] == model) & (combined["dataset"] == ds), "short"]
        l = combined.loc[(combined["model"] == model) & (combined["dataset"] == ds), "long"]
        row[f"{ds}_short"] = s.values[0] if len(s) else np.nan
        row[f"{ds}_long"] = l.values[0] if len(l) else np.nan
    rows.append(row)

wide_df = pd.DataFrame(rows).set_index("model")

col_tuples = []
for ds in ALL_DATASETS:
    col_tuples.append((ds, "short"))
    col_tuples.append((ds, "long"))
wide_df.columns = pd.MultiIndex.from_tuples(col_tuples, names=["dataset", "context"])

print("=" * 80)
print("PER-DATASET: answer_score -- Short (4k-32k) vs Long (64k-128k) averages")
print("=" * 80)
display(wide_df.round(4))

In [ ]:
# ============================================================================
# Per-Category: Short-Context vs Long-Context Average (answer_score)
# ============================================================================
# Short = 4k, 8k, 16k, 32k   |   Long = 64k, 96k, 128k

cat_short_avg = df_short.groupby(["model", "category"])["answer_score"].mean()
cat_long_avg = df_long.groupby(["model", "category"])["answer_score"].mean()

cat_combined = pd.DataFrame({"short": cat_short_avg, "long": cat_long_avg}).reset_index()

cat_order = [c for c in DATASETS_BY_CATEGORY.keys()]

rows = []
for model in models:
    row = {"model": model}
    for cat in cat_order:
        s = cat_combined.loc[(cat_combined["model"] == model) & (cat_combined["category"] == cat), "short"]
        l = cat_combined.loc[(cat_combined["model"] == model) & (cat_combined["category"] == cat), "long"]
        row[f"{cat}_short"] = s.values[0] if len(s) else np.nan
        row[f"{cat}_long"] = l.values[0] if len(l) else np.nan
    rows.append(row)

cat_wide_df = pd.DataFrame(rows).set_index("model")

col_tuples = []
for cat in cat_order:
    col_tuples.append((cat, "short"))
    col_tuples.append((cat, "long"))
cat_wide_df.columns = pd.MultiIndex.from_tuples(col_tuples, names=["category", "context"])

print("=" * 80)
print("PER-CATEGORY: answer_score -- Short (4k-32k) vs Long (64k-128k) averages")
print("=" * 80)
display(cat_wide_df.round(4))

## Per-Dataset x Context Length Breakdown

In [ ]:
# ============================================================================
# Per-Dataset x Context Length Breakdown
# ============================================================================
# Full detail table: mean of key metrics per (model, dataset, context_length)

agg_df = df.groupby(["model", "dataset", "context_length"])[METRICS].agg(["mean", "std"]).reset_index()

# Show answer_score by context length for each dataset (wide format)
print("ANSWER_SCORE by Model x Dataset x Context Length")
print("=" * 80)
ans_detail = df.groupby(["model", "dataset", "context_length_name"])["answer_score"].mean().reset_index()
ans_wide = ans_detail.pivot_table(index=["model", "dataset"], columns="context_length_name", values="answer_score")
# Reorder columns
ctx_order = [name for name, _ in CONTEXT_LENGTHS]
ctx_order = [c for c in ctx_order if c in ans_wide.columns]
ans_wide = ans_wide[ctx_order]
ans_wide["avg"] = ans_wide.mean(axis=1)
display(ans_wide.round(4))

## Per-Dataset Plots

For each metric, a grid of subplots (one per dataset) showing metric vs context length, with one line per model. Error bars show +/-1 std.

In [ ]:
# ============================================================================
# Per-Dataset Metric Plots (metric vs context length, one subplot per dataset)
# ============================================================================

PLOT_PER_DATASET = False  # Set True to render per-dataset plots

os.makedirs("plots", exist_ok=True)

# Stable colors per model
cmap = plt.get_cmap("tab10")
model_colors = {m: cmap(i % cmap.N) for i, m in enumerate(models)}

ctx_tick_labels = [name for name, _ in CONTEXT_LENGTHS]
ctx_tick_positions = [val for _, val in CONTEXT_LENGTHS]

# Which metrics to plot (skip citation metrics here)
plot_metrics = ["answer_score", "score", "gold_doc_overlap", "gold_doc_score",
                "has_answer", "truncated", "used_recall", "n_recall_spans",
                "correct_format", "correct_recall_usage", "recall_density_multiplier",
                "n_output_ids"]

if PLOT_PER_DATASET:
    for metric in plot_metrics:
        # Find datasets that have data for this metric
        valid_datasets = []
        for ds in ALL_DATASETS:
            sub = agg_df[agg_df["dataset"] == ds]
            if sub.empty:
                continue
            if (metric, "mean") not in sub.columns:
                continue
            if sub[(metric, "mean")].notna().any():
                valid_datasets.append(ds)

        if not valid_datasets:
            continue

        cols = min(3, len(valid_datasets))
        rows = math.ceil(len(valid_datasets) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows), squeeze=False)
        axes_flat = axes.flatten()

        # Determine shared y-axis range (skip for n_output_ids which has different scale)
        if metric != "n_output_ids":
            all_vals = []
            for ds in valid_datasets:
                sub = agg_df[agg_df["dataset"] == ds]
                upper = sub[(metric, "mean")] + sub[(metric, "std")].fillna(0)
                all_vals.extend(upper.dropna().tolist())
            max_y = max(all_vals) * 1.05 if all_vals else 1.0
        else:
            max_y = None

        plotted_models = set()
        for ax_idx, ds in enumerate(valid_datasets):
            ax = axes_flat[ax_idx]
            sub = agg_df[agg_df["dataset"] == ds]
            for m in models:
                m_df = sub[sub["model"] == m].sort_values("context_length")
                if m_df.empty:
                    continue
                plotted_models.add(m)
                ax.errorbar(
                    m_df["context_length"], m_df[(metric, "mean")],
                    yerr=m_df[(metric, "std")].fillna(0),
                    label=m, capsize=3, marker="o", markersize=3,
                    color=model_colors.get(m), linewidth=1.5,
                )
            metric_label = ANSWER_METRIC_NAMES.get(ds, metric) if metric == "answer_score" else metric.replace("_", " ").title()
            ax.set_title(f"{ds} -- {metric_label}", fontsize=11, fontweight="bold")
            ax.set_xlabel("Context Length")
            ax.set_ylabel(metric_label)
            ax.set_xticks(ctx_tick_positions)
            ax.set_xticklabels(ctx_tick_labels, rotation=45, ha="right", fontsize=8)
            if max_y is not None:
                ax.set_ylim(0, max_y)
            ax.grid(True, alpha=0.3, linestyle="--")

        for ax in axes_flat[len(valid_datasets):]:
            ax.axis("off")

        fig.suptitle(f"{metric.replace('_', ' ').title()} vs Context Length", fontsize=14, fontweight="bold", y=0.99)

        # Shared legend
        handles = [Line2D([0], [0], color=model_colors[m], marker="o", linestyle="-", label=m)
                   for m in models if m in plotted_models]
        if handles:
            fig.legend(handles, [h.get_label() for h in handles],
                       loc="upper center", bbox_to_anchor=(0.5, 0.96),
                       ncol=min(len(handles), 4), frameon=True, fontsize=8)

        fig.tight_layout(rect=[0, 0, 1, 0.93])
        fig.savefig(f"plots/{metric}.png", dpi=150, bbox_inches="tight")
        print(f"Saved plots/{metric}.png")
        plt.show()
else:
    print("Per-dataset plots skipped (PLOT_PER_DATASET = False)")

## Category-Level Summary Plots

Average `answer_score` and `score` per category x context length, with datasets within each category averaged together. One subplot per category.

In [ ]:
# ============================================================================
# Category-Level Summary Plots
# ============================================================================

PLOT_CATEGORY_SUMMARY = False  # Set True to render category summary plots

# Always compute cat_agg (used downstream)
cat_agg = df.groupby(["model", "category", "context_length"])["answer_score"].mean().reset_index()

if PLOT_CATEGORY_SUMMARY:
    for metric_name in ["answer_score", "score"]:
        cat_agg_plot = df.groupby(["model", "category", "context_length"])[metric_name].mean().reset_index()
        valid_cats = [c for c in DATASETS_BY_CATEGORY.keys() if c in cat_agg_plot["category"].unique()]

        if not valid_cats:
            continue

        cols = min(3, len(valid_cats))
        rows = math.ceil(len(valid_cats) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows), squeeze=False)
        axes_flat = axes.flatten()

        plotted_models = set()
        for ax_idx, cat in enumerate(valid_cats):
            ax = axes_flat[ax_idx]
            sub = cat_agg_plot[cat_agg_plot["category"] == cat]
            for m in models:
                m_df = sub[sub["model"] == m].sort_values("context_length")
                if m_df.empty:
                    continue
                plotted_models.add(m)
                ax.plot(m_df["context_length"], m_df[metric_name],
                        marker="o", markersize=4, linewidth=1.5,
                        color=model_colors.get(m), label=m)
            ax.set_title(cat, fontsize=12, fontweight="bold")
            ax.set_xlabel("Context Length")
            ax.set_ylabel(metric_name.replace("_", " ").title())
            ax.set_xticks(ctx_tick_positions)
            ax.set_xticklabels(ctx_tick_labels, rotation=45, ha="right", fontsize=8)
            ax.set_ylim(0, 1.05)
            ax.grid(True, alpha=0.3, linestyle="--")

        for ax in axes_flat[len(valid_cats):]:
            ax.axis("off")

        fig.suptitle(f"Category {metric_name.replace('_', ' ').title()} vs Context Length",
                     fontsize=14, fontweight="bold", y=0.99)
        handles = [Line2D([0], [0], color=model_colors[m], marker="o", linestyle="-", label=m)
                   for m in models if m in plotted_models]
        if handles:
            fig.legend(handles, [h.get_label() for h in handles],
                       loc="upper center", bbox_to_anchor=(0.5, 0.96),
                       ncol=min(len(handles), 4), frameon=True, fontsize=8)
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        fig.savefig(f"plots/category_{metric_name}.png", dpi=150, bbox_inches="tight")
        print(f"Saved plots/category_{metric_name}.png")
        plt.show()
else:
    print("Category summary plots skipped (PLOT_CATEGORY_SUMMARY = False)")

## Example Viewer

Filter by model, dataset, and context length. Shows metadata, scores, extracted answer, and full completion for qualitative analysis.

In [ ]:
# ============================================================================
# Example Viewer -- filter and inspect individual completions
# ============================================================================

# ---- FILTER SETTINGS (set to None to skip filtering) ----
filter_models = None            # e.g., ["recallm_qwen2"] or None for all
filter_datasets = None          # e.g., ["hotpotqa"] or None for all
filter_context_lengths = None   # e.g., [8000] or None for all
filter_categories = None        # e.g., ["icl"] or None for all
num_samples = 20                # Number of examples to display
random_seed = 42                # Set to None for no shuffling

# ---- FILTERING ----
fdf = df.copy()
if filter_models:
    fdf = fdf[fdf["model"].isin(filter_models)]
if filter_datasets:
    fdf = fdf[fdf["dataset"].isin(filter_datasets)]
if filter_context_lengths:
    fdf = fdf[fdf["context_length"].isin(filter_context_lengths)]
if filter_categories:
    fdf = fdf[fdf["category"].isin(filter_categories)]

if random_seed is not None:
    fdf = fdf.sample(frac=1, random_state=random_seed)
sample = fdf.head(num_samples)

print(f"Showing {len(sample)} of {len(fdf)} filtered examples")
print(f"Filters: models={filter_models}, datasets={filter_datasets}, "
      f"ctx={filter_context_lengths}, categories={filter_categories}\n")

# Score fields to show
score_fields = ["score", "answer_score", "gold_doc_overlap", "gold_doc_score",
                "has_answer", "truncated", "correct_format", "correct_recall_usage",
                "used_recall", "n_recall_spans", "recall_density_multiplier",
                "n_output_ids", "citation_f1_plain"]

for idx, (_, row) in enumerate(sample.iterrows()):
    print(f"\n{'=' * 90}")
    print(f"EXAMPLE {idx + 1}  |  {row['model']}  |  {row['dataset']}  |  {row['context_length_name']}  |  {row['category']}")
    print(f"{'=' * 90}")

    print(f"\n--- Question & Answer ---")
    print(f"Question: {row.get('question', 'N/A')}")
    print(f"Answer:   {row.get('answer', 'N/A')}")

    print(f"\n--- Scores ---")
    for sf in score_fields:
        val = row.get(sf)
        if val is None or (isinstance(val, float) and (np.isnan(val) or val == 0.0 and sf.startswith("citation"))):
            continue
        if isinstance(val, float):
            print(f"  {sf:>30s}: {val:.4f}")
        else:
            print(f"  {sf:>30s}: {val}")

    # Model answer (extracted)
    model_ans = row.get("model_answer")
    if model_ans:
        print(f"\n--- Model Answer (extracted) ---")
        print(model_ans[:500])

    # Full completion (truncated for readability)
    completion = row.get("completion", "")
    print(f"\n--- Completion ({len(completion)} chars) ---")
    print(completion[:3000])
    if len(completion) > 3000:
        print(f"\n... [truncated, {len(completion) - 3000} more chars]")
    print()